# LeetCode #24 — Swap Nodes in Pairs: nối lại con trỏ trên Linked List

**Mức độ:** Medium · **Chủ đề:** Linked List, Recursion

Skeleton trên LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def swapPairs(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Mục tiêu của notebook là luyện kỹ năng quan trọng nhất của Linked List: **thay đổi các mũi tên `next` theo đúng thứ tự** mà không làm "rơi mất" phần còn lại của danh sách.

---

## Sau notebook này, bạn sẽ nắm được

1. Vì sao đề **cấm đổi giá trị** và bắt phải đổi **nút**.
2. Cách vẽ sơ đồ "trước / sau" để biết cần sửa **những mũi tên nào**.
3. Thứ tự sửa mũi tên an toàn, dùng `dummy` và con trỏ `prev`.
4. Lời giải **đệ quy** gọn gàng.
5. Tổng quát: đảo từng nhóm `k` nút.

# 1. Bài toán là gì?

Cho một Linked List, **đổi chỗ từng cặp nút kề nhau** và trả về `head` mới.

**Bắt buộc:** không được sửa giá trị (`val`) trong nút — chỉ được thay đổi **chính các nút** (tức là các mũi tên `next`).

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 100]`
- `0 <= Node.val <= 100`

---

## Ví dụ

```text
[1, 2, 3, 4]  ->  [2, 1, 4, 3]
[]            ->  []
[1]           ->  [1]
[1, 2, 3]     ->  [2, 1, 3]      (nút lẻ cuối giữ nguyên)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Đệ quy (recursion)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) =

 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.3 Áp dụng vào bài này

Đổi chỗ từng cặp nút là bài **đổi con trỏ** trên danh sách liên kết: với `prev → a → b → nxt`, nối lại thành `prev → b → a → nxt` (nút giả làm `prev` cho cặp đầu). Cách **đệ quy**: đổi cặp đầu rồi để lời gọi đệ quy xử lý phần còn lại.

# 3. Nền tảng

In [11]:
from typing import Optional, List


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode]) -> List[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

print(to_list(build_list([1, 2, 3, 4])))

[1, 2, 3, 4]


## "Đổi giá trị" — dễ nhưng bị cấm

Cách lười nhất: đi từng cặp và đổi `val`. Kết quả **in ra** giống hệt, nhưng đề cấm vì:

- Trong thực tế, mỗi nút có thể chứa dữ liệu lớn, hoặc có nơi khác đang giữ tham chiếu tới nút đó — đổi giá trị sẽ làm sai lệch các tham chiếu ấy.
- Bài này muốn bạn luyện **thao tác con trỏ**.

In [12]:
def swap_values_forbidden(head: Optional[ListNode]) -> Optional[ListNode]:
    node = head
    while node and node.next:
        node.val, node.next.val = node.next.val, node.val
        node = node.next.next
    return head

head = build_list([1, 2, 3, 4])
first_node = head                      # giữ tham chiếu tới nút đầu tiên (giá trị 1)
result = swap_values_forbidden(head)
print(to_list(result), "| nút mà first_node trỏ tới giờ có val =", first_node.val)

[2, 1, 4, 3] | nút mà first_node trỏ tới giờ có val = 2


`first_node` từng là "nút số 1", giờ lại mang giá trị `2` — các tham chiếu bên ngoài bị sai. Đổi **nút** thì `first_node` vẫn mang giá trị `1`, chỉ là nó nằm ở vị trí thứ hai.

# 4. Vẽ sơ đồ: cần sửa những mũi tên nào?

Xét một cặp `a -> b` nằm giữa `prev` (nút đứng trước cặp) và `nxt` (nút đứng sau cặp):

```text
TRƯỚC:   prev -> a -> b -> nxt
SAU:     prev -> b -> a -> nxt
```

So sánh, có **3 mũi tên** thay đổi:

| Mũi tên | Trước | Sau |
|---|---|---|
| `prev.next` | `a` | `b` |
| `b.next` | `nxt` | `a` |
| `a.next` | `b` | `nxt` |

Thứ tự sửa **an toàn**: lưu `a`, `b`, `nxt` vào biến trước, rồi sửa tùy ý — vì đã có biến giữ, không nút nào bị "rơi mất".

```python
a = prev.next
b = a.next
nxt = b.next      # lưu lại TRƯỚC khi sửa b.next

a.next = nxt
b.next = a
prev.next = b
prev = a          # a giờ là nút cuối của cặp, đứng trước cặp kế tiếp
```

In [13]:
head = build_list([1, 2, 3, 4])
dummy = ListNode(0, head)
prev = dummy

a = prev.next          # 1
b = a.next             # 2
nxt = b.next           # 3
a.next = nxt
b.next = a
prev.next = b
print("Sau khi đổi cặp đầu:", to_list(dummy.next))

Sau khi đổi cặp đầu: [2, 1, 3, 4]


## Vì sao cần `dummy`?

Cặp **đầu tiên** không có `prev` thật. `dummy` đóng vai `prev` cho cặp đầu, và cuối cùng `dummy.next` chính là `head` mới (nút `2`). Không cần `if` riêng cho cặp đầu.

# 5. Lời giải chính: lặp

```text
prev = dummy
while còn ít nhất 2 nút sau prev:
    đổi cặp (prev.next, prev.next.next)
    prev = nút cuối của cặp vừa đổi
```

In [14]:
def swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy

    while prev.next and prev.next.next:
        a = prev.next
        b = a.next
        nxt = b.next

        a.next = nxt
        b.next = a
        prev.next = b

        prev = a

    return dummy.next

for vals in [[1, 2, 3, 4], [], [1], [1, 2, 3]]:
    print(vals, "->", to_list(swap_pairs(build_list(vals))))

[1, 2, 3, 4] -> [2, 1, 4, 3]
[] -> []
[1] -> [1]
[1, 2, 3] -> [2, 1, 3]


# 6. Mô phỏng từng bước

In [15]:
def trace_swap_pairs(values: List[int]):
    head = build_list(values)
    dummy = ListNode(0, head)
    prev = dummy
    step = 1
    print("Ban đầu:", to_list(dummy.next))
    while prev.next and prev.next.next:
        a, b = prev.next, prev.next.next
        nxt = b.next
        prev_name = "dummy" if prev is dummy else prev.val
        print(f"Cặp {step}: prev = {prev_name}, a = {a.val}, b = {b.val}, nxt = {nxt.val if nxt else None}")
        a.next = nxt
        b.next = a
        prev.next = b
        print(f"   sau khi đổi: {to_list(dummy.next)}")
        prev = a
        step += 1
    reason = "không còn nút nào" if prev.next is None else f"chỉ còn 1 nút ({prev.next.val})"
    print(f"Dừng vì {reason}")
    return dummy.next

_ = trace_swap_pairs([1, 2, 3, 4, 5])

Ban đầu: [1, 2, 3, 4, 5]
Cặp 1: prev = dummy, a = 1, b = 2, nxt = 3
   sau khi đổi: [2, 1, 3, 4, 5]
Cặp 2: prev = 1, a = 3, b = 4, nxt = 5
   sau khi đổi: [2, 1, 4, 3, 5]
Dừng vì chỉ còn 1 nút (5)


# 7. Cách 2: đệ quy

Quan sát: đổi cặp đầu `(a, b)`, phần còn lại (từ nút thứ 3) chính là **cùng bài toán, nhỏ hơn**.

```python
def swap(head):
    if head is None or head.next is None:   # base case: 0 hoặc 1 nút
        return head
    a, b = head, head.next
    a.next = swap(b.next)                   # đổi phần còn lại, nối vào sau a
    b.next = a
    return b                                # b là head mới
```

In [16]:
def swap_pairs_recursive(head: Optional[ListNode]) -> Optional[ListNode]:
    if head is None or head.next is None:
        return head
    a, b = head, head.next
    a.next = swap_pairs_recursive(b.next)
    b.next = a
    return b

for vals in [[1, 2, 3, 4], [], [1], [1, 2, 3], [1, 2, 3, 4, 5, 6]]:
    print(vals, "->", to_list(swap_pairs_recursive(build_list(vals))))

[1, 2, 3, 4] -> [2, 1, 4, 3]
[] -> []
[1] -> [1]
[1, 2, 3] -> [2, 1, 3]
[1, 2, 3, 4, 5, 6] -> [2, 1, 4, 3, 6, 5]


Đệ quy gọn hơn nhưng dùng `O(n)` bộ nhớ cho call stack (`n/2` lời gọi lồng nhau). Với `n ≤ 100` không sao; nhưng với danh sách rất dài, Python có giới hạn độ sâu đệ quy (mặc định khoảng 1000).

# 8. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def swapPairs(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        prev = dummy
        while prev.next and prev.next.next:
            a = prev.next
            b = a.next
            a.next = b.next
            b.next = a
            prev.next = b
            prev = a
        return dummy.next

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


sol = Solution()
run = lambda vals: to_list(sol.swapPairs(build_list(vals)))

check("Ví dụ 1", run([1, 2, 3, 4]), [2, 1, 4, 3])
check("Ví dụ 2: rỗng", run([]), [])
check("Ví dụ 3: một nút", run([1]), [1])
check("Số nút lẻ", run([1, 2, 3]), [2, 1, 3])
check("Hai nút", run([5, 9]), [9, 5])
check("Giá trị trùng", run([7, 7, 7, 7]), [7, 7, 7, 7])

Ví dụ 1                PASS  [2, 1, 4, 3]
Ví dụ 2: rỗng          PASS  []
Ví dụ 3: một nút       PASS  [1]
Số nút lẻ              PASS  [2, 1, 3]
Hai nút                PASS  [9, 5]
Giá trị trùng          PASS  [7, 7, 7, 7]


## Kiểm tra: đổi **nút**, không đổi giá trị

Ta ghi lại `id` (danh tính) của từng nút trước khi đổi. Sau khi đổi, thứ tự các **nút** phải đúng là thứ tự đổi cặp, và mỗi nút giữ nguyên `val` của nó.

In [19]:
import random

def expected_order(values):
    out = values[:]
    for i in range(0, len(out) - 1, 2):
        out[i], out[i + 1] = out[i + 1], out[i]
    return out

random.seed(24)
for _ in range(2000):
    vals = [random.randint(0, 100) for _ in range(random.randint(0, 20))]
    head = build_list(vals)
    nodes = []
    node = head
    while node:
        nodes.append(node)
        node = node.next
    original_vals = {id(nd): nd.val for nd in nodes}

    new_head = sol.swapPairs(head)
    new_nodes = []
    node = new_head
    while node:
        new_nodes.append(node)
        node = node.next

    assert [id(nd) for nd in new_nodes] == [id(nd) for nd in expected_order(nodes)]
    assert all(nd.val == original_vals[id(nd)] for nd in new_nodes)
    assert to_list(swap_pairs_recursive(build_list(vals))) == expected_order(vals)
print("2000 test: đổi đúng NÚT, giá trị mỗi nút giữ nguyên; bản đệ quy cũng khớp")

2000 test: đổi đúng NÚT, giá trị mỗi nút giữ nguyên; bản đệ quy cũng khớp


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến (bản lặp):** ở đầu mỗi vòng, đoạn từ `dummy` tới `prev` đã được đổi cặp xong, và `prev.next` là nút đầu của phần **chưa xử lý** (còn nguyên thứ tự gốc).

- Mỗi vòng đổi đúng một cặp và nối nó vào sau `prev`, rồi dời `prev` tới cuối cặp — bất biến được giữ.
- Vòng lặp dừng khi phần chưa xử lý có 0 hoặc 1 nút — những nút này giữ nguyên, đúng yêu cầu.

**Độ phức tạp** (`n` là số nút):

```text
Bản lặp:   Time O(n), Space O(1)
Bản đệ quy: Time O(n), Space O(n) cho call stack
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Sửa mũi tên sai thứ tự mà không lưu biến

Nếu viết `b.next = a` **trước** khi lưu `b.next` (tức `nxt`), phần còn lại của danh sách bị **mất**:

In [20]:
def wrong_order(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a = prev.next
        b = a.next
        b.next = a                  # SAI: ghi đè b.next trước khi lưu nxt
        a.next = b.next             # b.next giờ là a -> a trỏ vào chính nó!
        prev.next = b
        prev = a
        break                       # dừng ngay để không lặp vô hạn trong demo
    return dummy.next

head = wrong_order(build_list([1, 2, 3, 4]))
print("head =", head.val, "| head.next =", head.next.val, "| head.next.next =", head.next.next.val,
      "-> nút 1 trỏ vào chính nó, mất nút 3, 4")

head = 2 | head.next = 1 | head.next.next = 1 -> nút 1 trỏ vào chính nó, mất nút 3, 4


## Lỗi 2: Quên dời `prev`

Không có `prev = a`, vòng sau lại đổi cặp ở **cùng vị trí** → đổi qua đổi lại hoặc lặp vô hạn.

## Lỗi 3: Dời `prev` tới `b` thay vì `a`

Sau khi đổi, thứ tự là `prev -> b -> a -> nxt`. Nút đứng **trước cặp kế tiếp** là `a`, không phải `b`.

In [21]:
def wrong_prev_to_b(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a = prev.next
        b = a.next
        a.next = b.next
        b.next = a
        prev.next = b
        prev = b                    # SAI: phải là a
    return dummy.next

print("Sai :", to_list(wrong_prev_to_b(build_list([1, 2, 3, 4, 5, 6]))))
print("Đúng:", to_list(swap_pairs(build_list([1, 2, 3, 4, 5, 6]))))

Sai : [2, 3, 4, 5, 6, 1]
Đúng: [2, 1, 4, 3, 6, 5]


## Lỗi 4: Điều kiện lặp `while prev.next`

Khi chỉ còn 1 nút, `prev.next.next` là `None` → `b` là `None` → `b.next` gây `AttributeError`. Phải kiểm tra **cả hai** nút.

# 11. Mẹo nhận diện pattern

**Sửa con trỏ trên Linked List:**

1. **Vẽ sơ đồ trước / sau**, liệt kê các mũi tên thay đổi.
2. **Lưu mọi nút cần dùng** vào biến trước khi sửa.
3. Dùng **dummy** nếu `head` có thể thay đổi.
4. Xác định rõ con trỏ `prev` phải đứng ở đâu cho vòng sau.

Bài cùng họ: **#206 Reverse Linked List**, **#92 Reverse Linked List II**, **#25 Reverse Nodes in k-Group** (Hard — tổng quát của bài này), **#143 Reorder List**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[10, 20, 30, 40, 50]`, viết giá trị của `prev`, `a`, `b`, `nxt` ở mỗi vòng, và danh sách sau mỗi vòng.

## Bài 2 — Đảo ngược toàn bộ danh sách (LeetCode #206)

Đây là bài nền tảng nhất về sửa con trỏ. Đảo ngược danh sách bằng vòng lặp, `O(1)` bộ nhớ.

```python
reverse([1, 2, 3, 4, 5]) -> [5, 4, 3, 2, 1]
reverse([])              -> []
```

In [22]:
def exercise_2_reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO: dùng prev = None, cur = head; mỗi bước lưu nxt = cur.next, đảo cur.next = prev, dời prev, cur
    pass


# assert to_list(exercise_2_reverse_list(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
# assert to_list(exercise_2_reverse_list(build_list([]))) == []
# assert to_list(exercise_2_reverse_list(build_list([1]))) == [1]

## Bài 3 — Điền chỗ trống: Swap Pairs

In [23]:
def exercise_3_swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        # TODO 1: lấy a, b
        # TODO 2: sửa 3 mũi tên: a.next, b.next, prev.next
        # TODO 3: dời prev
        pass
    return dummy.next


# assert to_list(exercise_3_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
# assert to_list(exercise_3_swap_pairs(build_list([1, 2, 3]))) == [2, 1, 3]

## Bài 4 — Biến thể: đổi chỗ phần tử đầu và cuối

Đổi chỗ **nút** đầu và **nút** cuối (không đổi giá trị). Danh sách 0 hoặc 1 nút giữ nguyên.

```python
swap_first_last([1, 2, 3, 4]) -> [4, 2, 3, 1]
swap_first_last([1, 2])       -> [2, 1]
```

In [24]:
def exercise_4_swap_first_last(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_4_swap_first_last(build_list([1, 2, 3, 4]))) == [4, 2, 3, 1]
# assert to_list(exercise_4_swap_first_last(build_list([1, 2]))) == [2, 1]
# assert to_list(exercise_4_swap_first_last(build_list([1]))) == [1]
# assert to_list(exercise_4_swap_first_last(build_list([]))) == []

## Bài 5 — Mini challenge: đảo từng nhóm `k` nút (LeetCode #25)

Đảo ngược từng nhóm `k` nút liên tiếp. Nhóm cuối nếu **không đủ `k` nút** thì giữ nguyên. Với `k = 2`, đây chính là Swap Pairs.

```text
[1, 2, 3, 4, 5], k = 2  ->  [2, 1, 4, 3, 5]
[1, 2, 3, 4, 5], k = 3  ->  [3, 2, 1, 4, 5]
```

In [25]:
def exercise_5_reverse_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3]), 1)) == [1, 2, 3]

# 13. Gợi ý

**Bài 1**
- Mức 1: có 2 cặp được đổi, nút 50 giữ nguyên.

**Bài 2**
- Mức 1: mỗi bước đảo **một** mũi tên `cur.next`.
- Mức 2: `nxt = cur.next; cur.next = prev; prev = cur; cur = nxt`.
- Mức 3: khi `cur` là `None`, `prev` là head mới.

**Bài 4**
- Mức 1: tìm nút **áp chót** `before_last` và nút cuối `last`.
- Mức 2: trường hợp 2 nút cần xử lý riêng (đầu và cuối kề nhau).
- Mức 3: trường hợp ≥ 3 nút: `last.next = head.next`, `before_last.next = head`, `head.next = None`, trả `last`.

**Bài 5**
- Mức 1: trước khi đảo, kiểm tra còn đủ `k` nút không (đi thử `k` bước).
- Mức 2: đảo `k` nút như Bài 2, rồi nối đầu/cuối nhóm vào `prev` và phần sau.
- Mức 3: sau khi đảo, nút **đầu cũ** của nhóm thành nút **cuối**, dùng nó làm `prev` cho nhóm sau.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
_ = trace_swap_pairs([10, 20, 30, 40, 50])

Ban đầu: [10, 20, 30, 40, 50]
Cặp 1: prev = dummy, a = 10, b = 20, nxt = 30
   sau khi đổi: [20, 10, 30, 40, 50]
Cặp 2: prev = 10, a = 30, b = 40, nxt = 50
   sau khi đổi: [20, 10, 40, 30, 50]
Dừng vì chỉ còn 1 nút (50)


## Đáp án Bài 2

In [27]:
def answer_2_reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

assert to_list(answer_2_reverse_list(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(answer_2_reverse_list(build_list([]))) == []
assert to_list(answer_2_reverse_list(build_list([1]))) == [1]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [28]:
def answer_3_swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a = prev.next
        b = a.next
        a.next = b.next
        b.next = a
        prev.next = b
        prev = a
    return dummy.next

assert to_list(answer_3_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
assert to_list(answer_3_swap_pairs(build_list([1, 2, 3]))) == [2, 1, 3]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_swap_first_last(head: Optional[ListNode]) -> Optional[ListNode]:
    if head is None or head.next is None:
        return head
    if head.next.next is None:                 # đúng 2 nút
        second = head.next
        second.next = head
        head.next = None
        return second
    before_last = head
    while before_last.next.next:
        before_last = before_last.next
    last = before_last.next
    last.next = head.next
    before_last.next = head
    head.next = None
    return last

assert to_list(answer_4_swap_first_last(build_list([1, 2, 3, 4]))) == [4, 2, 3, 1]
assert to_list(answer_4_swap_first_last(build_list([1, 2]))) == [2, 1]
assert to_list(answer_4_swap_first_last(build_list([1]))) == [1]
assert to_list(answer_4_swap_first_last(build_list([]))) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_reverse_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while True:
        # kiểm tra còn đủ k nút
        check_node = prev
        for _ in range(k):
            check_node = check_node.next
            if check_node is None:
                return dummy.next
        group_head = prev.next
        after = check_node.next
        # đảo k nút
        p, cur = after, group_head
        for _ in range(k):
            nxt = cur.next
            cur.next = p
            p = cur
            cur = nxt
        prev.next = p              # p là đầu mới của nhóm
        prev = group_head          # đầu cũ giờ là cuối nhóm

assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3]), 1)) == [1, 2, 3]
for _ in range(500):
    vals = [random.randint(0, 9) for _ in range(random.randint(0, 15))]
    assert to_list(answer_5_reverse_k_group(build_list(vals), 2)) == expected_order(vals)
print("Bài 5 PASS (với k = 2 khớp Swap Pairs trên 500 test)")

Bài 5 PASS (với k = 2 khớp Swap Pairs trên 500 test)


Mẹo trong đáp án: khi đảo nhóm, khởi tạo `p = after` (không phải `None`) — nút cuối nhóm sau khi đảo sẽ tự trỏ tới phần còn lại.

# 15. Tổng kết

> Vẽ **trước / sau** → liệt kê mũi tên cần sửa → **lưu nút vào biến trước** → sửa → dời `prev` tới **cuối cặp vừa đổi** (`a`). Dùng `dummy` để cặp đầu không cần xử lý riêng.

---

## Checklist trước khi nộp

- [ ] Không đổi `val`, chỉ đổi `next`?
- [ ] Điều kiện lặp kiểm tra **hai** nút: `prev.next and prev.next.next`?
- [ ] Sửa đủ 3 mũi tên: `a.next`, `b.next`, `prev.next`?
- [ ] `prev = a` (không phải `b`)?
- [ ] Trả về `dummy.next`?
- [ ] Đã thử rỗng, 1 nút, số nút lẻ?